In [135]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

In [142]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [143]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [95]:
# Import datasets
# datasets = {}
# dataset_names = ['clfever', 'phemeplus', 'vitc']
# for dataset_name in dataset_names:
#     with open(f'{dataset_name}.json') as f:
#         datasets[dataset_name] = json.load(f)

In [144]:
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [130]:
# Create batches of 25 claims
vitc_batches = {}
for i in range(0, len(vitc), 25):
    batch_dataset = vitc[i:i+25]
    claim_ids = []
    for batch_datapoint in batch_dataset:
        id = batch_datapoint['claim_id']
        claim_ids.append(id)
    batch_id = f'batch_vitc_{i//25 + 1}'
    vitc_batches[batch_id] = claim_ids

# Create an extra 8 batches to confirm the annotations
# confirm_batches = {f'batch_vitc_confirm_{i}': [] for i in range(1, 5)}

# for i in range(1,5):
    
# for batch_id in batches.keys():
#     claim_ids = batches[batch_id]
#     for i in range(1,5):
#         confirm_batch_id = f'batch_vitc_confirm_{i}'
#         confirm_batches[confirm_batch_id].append(claim_ids[8 * (i-1)])


In [145]:
# Populate Vercel KV with vitc datasets
# for datapoint in vitc:
#     id = datapoint['claim_id']
#     r.hset(id, mapping={
#         'claim': datapoint['claim'],
#         'evidence': datapoint['evidence'],
#         'label': datapoint['label']
#     })

# Populate Vercel KV with vitc batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

# Populate Vercel KV with confirm batches



In [96]:
# Populate Vercel KV with datasets
# for dataset_name in dataset_names:
#     dataset = datasets[dataset_name]
#     for datapoint in dataset:
#         id = datapoint['claim_id']
        # r.hset(id, mapping={
        #     'claim': datapoint['claim'],
        #     'evidence': datapoint['evidence'],
        #     'label': datapoint['label']
        # })

In [54]:
# Create batches containing 25 datapoints each
# batch_ids = []
# for dataset_name in dataset_names:
#     dataset = datasets[dataset_name]
#     for i in range(0, len(dataset), 25):
#         batch_dataset = dataset[i:i+25]
#         claim_ids = []
#         for batch_datapoint in batch_dataset:
#             id = batch_datapoint['claim_id']
#             claim_ids.append(id)
#         batch_id = f'batch_{dataset_name}_{i//25 + 1}'
#         batch_ids.append(batch_id)
#         r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})   

In [107]:
# Import assessesment samples
vitc_assessment = pd.read_csv('Assesment samples - VitC.csv')

In [28]:
vitc_assessment

,Id,claim,evidence,Ground Truth,Label,Unnamed: 5
0,asses_1_vitc,There have been more than five confirmed cases...,"On 25 January 2020 , the number of laboratory-...",SUPPORTS,d,NaN
1,asses_2_vitc,The most prominent smartphone vendor in the wo...,BlackBerry was one of the most prominent smart...,SUPPORTS,a,This one is in the guideline
2,asses_3_vitc,"According to expectations , the Boeing B-52 St...","After being upgraded between 2013 and 2015 , i...",REFUTES,d,NaN
3,asses_4_vitc,Marcus Bentley is a British chef .,"Marcus Morgan Bentley ( born October 4 , 1967 ...",SUPPORTS,d,NaN
4,asses_5_vitc,"Before March 29 , 2020 , Nevada had less than ...","As of March 29 , 2020 , 738 positive cases and...",REFUTES,a,NaN


In [30]:
# iterate through rows of dataframe
assess_ids = []
for index, row in vitc_assessment.iterrows():
    id = row["Id"]
    assess_ids.append(id)
    claim = row["claim"]
    evidence = row["evidence"]
    label = row["Ground Truth"]
    reasoning = row["Label"]
    if reasoning == 'a':
        reasoning = 'abductive'
    elif reasoning == 'd':
        reasoning = 'deductive'
    
    r.hset(id, mapping={
        'claim': claim,
        'evidence': evidence,
        'label': label,
        'reasoning': reasoning
    })


In [108]:
# create batch for assessement
r.hset('assess_vitc', mapping={'claim_ids': json.dumps(assess_ids)}) 

ConnectionError: Error 54 connecting to model-bedbug-21479.upstash.io:6379. Connection reset by peer.

In [34]:
r.hgetall('assess_vitc')

{b'claim_ids': b'["asses_1_vitc", "asses_2_vitc", "asses_3_vitc", "asses_4_vitc", "asses_5_vitc"]'}

In [146]:
# Delete queue 
r.delete('queue')

1

In [132]:
vitc_only_batch = [batch_id for batch_id in vitc_batches.keys()]

In [148]:
# Create queue 
r.lpush('queue', *vitc_only_batch)

20

In [150]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

KeyboardInterrupt: 

In [139]:
r.hgetall('batch_vitc_20')

{b'claim_ids': b'["vitc_16915", "vitc_15819", "vitc_8186", "vitc_12391", "vitc_3532", "vitc_16455", "vitc_5317", "vitc_13761", "vitc_17114", "vitc_9659", "vitc_16073", "vitc_14246", "vitc_17686", "vitc_8802", "vitc_12141", "vitc_18578", "vitc_4565", "vitc_18936", "vitc_8683", "vitc_5167", "vitc_17640", "vitc_16311", "vitc_1159", "vitc_4783", "vitc_17988"]'}

In [60]:
# delete participants
# r.delete('participants')

1

In [89]:
# Get list og participants who completed the task
r.lrange('participants',0,-1)

[b'{"participant":"Talia","batchId":"batch_vitc_16","stage":"assessment"}',
 b'{"participant":"Mobile tester","batchId":"batch_vitc_17","stage":"assessment"}',
 b'{"participant":"tester","batchId":"batch_vitc_15","stage":"assessment"}',
 b'{"participant":"Panzerschokoladelover1945","batchId":"batch_vitc_17","stage":"assessment"}',
 b'{"participant":"Panzerschokoladelover1945","batchId":"batch_vitc_18","stage":"assessment"}',
 b'{"participant":"Test_Production_Success","batchId":"batch_vitc_1","stage":"annotation"}',
 b'{"participant":"Test_Production_Success","batchId":"batch_vitc_1","stage":"assessment"}',
 b'{"participant":"Test_Production_Fail","batchId":"batch_vitc_2","stage":"assessment"}',
 b'{"participant":"test_success","batchId":"batch_vitc_6","stage":"annotation"}',
 b'{"participant":"test_success","batchId":"batch_vitc_6","stage":"assessment"}']

In [81]:
# get answers from specific participant
r.hgetall('Test_Production_Success')

{b'vitc_19121': b'deductive',
 b'asses_5_vitc': b'abductive',
 b'vitc_7554': b'deductive',
 b'vitc_5355': b'deductive',
 b'vitc_9608': b'abductive',
 b'vitc_599': b'abductive',
 b'vitc_7545': b'abductive',
 b'vitc_4238': b'deductive',
 b'vitc_16758': b'deductive',
 b'vitc_13128': b'deductive',
 b'vitc_13018': b'abductive',
 b'asses_3_vitc': b'deductive',
 b'vitc_2812': b'abductive',
 b'vitc_18256': b'deductive',
 b'vitc_18632': b'deductive',
 b'vitc_6100': b'abductive',
 b'asses_2_vitc': b'abductive',
 b'asses_4_vitc': b'deductive',
 b'vitc_11740': b'deductive',
 b'vitc_13315': b'deductive',
 b'vitc_13729': b'abductive',
 b'vitc_7223': b'deductive',
 b'vitc_17092': b'deductive',
 b'vitc_10829': b'abductive',
 b'vitc_11888': b'abductive',
 b'vitc_4635': b'abductive',
 b'vitc_8582': b'deductive',
 b'asses_1_vitc': b'deductive',
 b'vitc_13850': b'deductive',
 b'vitc_320': b'abductive'}